# ket_operations_dict_based
Dictionary-only mixed-state format (pure states remain list-based).


In [ ]:
from itertools import product
from sympy import symbols, Rational, sqrt, pi, I, exp, cos, sin, Mod, simplify, conjugate, Matrix, zeros, IndexedBase

# ---------- Pure state helpers (unchanged visible format) ----------

def add_kets(*kets):
    coeffs = {}
    for ket in kets:
        for coeff, basis in ket:
            coeffs[basis] = coeffs.get(basis, 0) + coeff
    return sorted([(c, b) for b, c in coeffs.items() if c != 0], key=lambda x: x[1])

def _basis_sort_key(item):
    return item[1]

def _normalize_global_phase(state):
    norm, st = normalize(state)
    if norm == 0:
        return state
    return st

def _pure_state_key(state, phase_invariant=True):
    st = add_kets(state)
    if phase_invariant:
        st = _normalize_global_phase(st)
    return tuple(sorted(st, key=_basis_sort_key))

def tensor_product(state1, state2):
    return [(c1*c2, b1+b2) for c1,b1 in state1 for c2,b2 in state2]

def comp_st(*states):
    if not states:
        return [(1, ())]
    out = states[0]
    for s in states[1:]:
        out = tensor_product(out, s)
    return out

def create_zero_state(N):
    return [(1, tuple(0 for _ in range(N)))]

def create_bell_state(dim, shift, phase):
    omega = exp(2*pi*I/dim)
    return [(1/sqrt(dim) * omega**(k*phase), (k, Mod(k+shift, dim))) for k in range(dim)]

# ---------- Mixed state dict format ----------
# Dict[ pure_state_key(tuple[(coeff,basis), ...]), probability ]

def pure_to_mixed(pure_state, phase_invariant=True):
    return {_pure_state_key(pure_state, phase_invariant=phase_invariant): 1}

def merge_mixed_states(mixed_state, phase_invariant=True):
    merged = {}
    for key, prob in mixed_state.items():
        canon = _pure_state_key(list(key), phase_invariant=phase_invariant)
        merged[canon] = merged.get(canon, 0) + prob
    return {k: v for k, v in merged.items() if v != 0}

def comp_mixed_states(*mixed_states, merge=True, phase_invariant=True):
    if not mixed_states:
        return {}
    out = {}
    for combo in product(*[list(ms.items()) for ms in mixed_states]):
        prob = 1
        ket = [(1, ())]
        for state_key, p in combo:
            prob *= p
            ket = tensor_product(ket, list(state_key))
        k = _pure_state_key(ket, phase_invariant=phase_invariant)
        out[k] = out.get(k, 0) + prob
    if merge:
        out = merge_mixed_states(out, phase_invariant=phase_invariant)
    return out

# ---------- Gates on pure states ----------
def pauli_x(n, state):
    return [(c, tuple(b[:n] + (1-b[n],) + b[n+1:])) for c,b in state]

def pauli_z(n, state):
    return [((-c if b[n] == 1 else c), b) for c,b in state]

def pauli_y(n, state):
    # Y = i X Z (explicitly)
    return [(I*c, tuple(b[:n] + (1-b[n],) + b[n+1:])) for c,b in state if b[n]==0] + [(-I*c, tuple(b[:n] + (1-b[n],) + b[n+1:])) for c,b in state if b[n]==1]

def hadamard(n, state):
    s = 1/sqrt(2)
    out = []
    for c,b in state:
        if b[n] == 0:
            out.append((s*c, tuple(b[:n]+(0,)+b[n+1:])))
            out.append((s*c, tuple(b[:n]+(1,)+b[n+1:])))
        else:
            out.append((s*c, tuple(b[:n]+(0,)+b[n+1:])))
            out.append((-s*c, tuple(b[:n]+(1,)+b[n+1:])))
    return add_kets(out)

def rotation_x(n, angle, state):
    a = [(cos(angle/2)*c, b) for c,b in state]
    b = [(-I*sin(angle/2)*c, tuple(bs[:n]+(1-bs[n],)+bs[n+1:])) for c,bs in state]
    return add_kets(a+b)

def cnot(control, target, state):
    out=[]
    for c,b in state:
        t = b[target] if b[control]==0 else 1-b[target]
        out.append((c, b[:target]+(t,)+b[target+1:]))
    return out

def cz(control, target, state):
    return [(c if not (b[control]==1 and b[target]==1) else -c, b) for c,b in state]

def id(state):
    return state

# ---------- Mixed-state transforms ----------
def apply_gate_to_mixed_state(gate_function, params, mixed_state, merge=True, phase_invariant=True):
    out = {}
    for key, prob in mixed_state.items():
        new_ket = gate_function(*params, list(key))
        k = _pure_state_key(new_ket, phase_invariant=phase_invariant)
        out[k] = out.get(k, 0) + prob
    if merge:
        out = merge_mixed_states(out, phase_invariant=phase_invariant)
    return out

def scale_state(factor, state):
    return [(factor*c, b) for c,b in state]

def normalize(state):
    norm = sum(abs(c)**2 for c,_ in state)
    if norm == 0:
        return 0, state
    phase = 1
    for c,_ in state:
        if c != 0:
            phase = c/abs(c)
            break
    return norm, scale_state(1/sqrt(norm)*conjugate(phase), state)

def measurement(meas, indices, state):
    out=[]
    for mc,mb in meas:
        cmc = mc.conjugate()
        for sc,sb in state:
            if tuple(sb[i] for i in indices) == mb:
                out.append((cmc*sc, tuple(sb[i] for i in range(len(sb)) if i not in indices)))
    return add_kets(out)

def mixed_measurement(mixed_meas, indices, mixed_state, merge=True, phase_invariant=True):
    """Projective measurement on a mixed state WITHOUT output normalization.
    Returned probabilities are unnormalized branch weights, so their sum is the success probability.
    Use normalized_mixed_state(...) afterward if normalized output is desired.
    """
    out = {}
    for meas_key, p_m in mixed_meas.items():
        meas = list(meas_key)
        for state_key, p_s in mixed_state.items():
            projected = measurement(meas, indices, list(state_key))
            if projected:
                n2 = sum(abs(c)**2 for c, _ in projected)
                if n2 != 0:
                    k = _pure_state_key(projected, phase_invariant=phase_invariant)
                    out[k] = out.get(k, 0) + p_m * p_s * n2
    if merge:
        out = merge_mixed_states(out, phase_invariant=phase_invariant)
    return out

def add_operators(*ops, phase_invariant=True):
    out={}
    for op in ops:
        for k,v in op.items():
            canon = _pure_state_key(list(k), phase_invariant=phase_invariant)
            out[canon] = out.get(canon, 0) + v
    return {k:v for k,v in out.items() if v != 0}

def normalized_mixed_state(mixed_state):
    tr = sum(mixed_state.values())
    if tr == 0:
        return 0, mixed_state
    return tr, {k: v/tr for k,v in mixed_state.items()}

def fidelity(pure_state, mixed_state):
    total = 0
    idx = list(range(len(pure_state[0][1])))
    for k,p in mixed_state.items():
        proj = measurement(pure_state, idx, list(k))
        total += p * sum(abs(c)**2 for c,_ in proj)
    return simplify(total)

def infer_num_qubits(pure_state):
    for _,b in pure_state:
        return len(b)
    return 0

def mixed_to_DM(mixed_state):
    any_key = next(iter(mixed_state.keys()))
    n = infer_num_qubits(list(any_key))
    dim = 2**n
    rho = zeros(dim, dim)
    def idx(b):
        return int(''.join(map(str,b)), 2)
    for key,p in mixed_state.items():
        st = list(key)
        for c1,b1 in st:
            for c2,b2 in st:
                rho[idx(b1), idx(b2)] += p*c1*c2.conjugate()
    return simplify(rho)

def partial_transpose(rho, d=2, n=2, subsystems_to_transpose=[0]):
    dim = d**n
    if rho.shape != (dim, dim):
        raise ValueError(f'Expected {(dim, dim)}, got {rho.shape}')
    out = zeros(dim, dim)
    for col in range(dim):
        for row in range(dim):
            col_idx = [(col // d**(n-1-i)) % d for i in range(n)]
            row_idx = [(row // d**(n-1-i)) % d for i in range(n)]
            for k in subsystems_to_transpose:
                row_idx[k], col_idx[k] = col_idx[k], row_idx[k]
            nr = sum(row_idx[i] * d**(n-1-i) for i in range(n))
            nc = sum(col_idx[i] * d**(n-1-i) for i in range(n))
            out[nr, nc] = rho[row, col]
    return out

def eigenvalues(dm, symbolic_mode=True):
    vals = dm.eigenvals(multiple=True)
    return vals if symbolic_mode else sorted(vals, key=lambda x: x.evalf())

def display_mixed_state(mixed_state, label=None):
    if label:
        print(label + ':')
    for i,(k,p) in enumerate(mixed_state.items()):
        print(f'  [{i}] Prob: {p}, State: {list(k)}')

# ---------- Built-in Pauli probability helpers ----------

def make_general_prob(symbol='Pr'):
    Pr = IndexedBase(symbol)
    return lambda *bits: Pr[bits]

def make_iid_depolarizing_prob(F):
    def p2(a,b):
        return F if (a,b)==(0,0) else (1-F)/3
    return lambda *bits: simplify(__import__('functools').reduce(lambda x,y: x*y, [p2(bits[2*i], bits[2*i+1]) for i in range(len(bits)//2)], 1))

def make_bitflip_prob(p):
    # only X errors, no Z: b must be 0 for all qubits
    def f(*bits):
        k = len(bits)//2
        out = 1
        for i in range(k):
            a,b = bits[2*i], bits[2*i+1]
            if b != 0:
                return 0
            out *= (p if a==1 else (1-p))
        return simplify(out)
    return f

def make_pauli_table_prob(table):
    # table maps (a,b) -> prob expression
    return lambda *bits: simplify(__import__('functools').reduce(lambda x,y: x*y, [table[(bits[2*i], bits[2*i+1])] for i in range(len(bits)//2)], 1))

# ---------- General Pauli noise ----------
def pauli_noise(state, noisy_qubits, prob_func=None, merge=True, phase_invariant=True):
    """
    Apply symbolic Pauli noise on selected qubits.
    - state: pure_state (list) OR mixed_state (dict)
    - noisy_qubits: iterable of qubit indices
    - prob_func(*bits): probability for pattern bits (a1,b1,...,ak,bk)
    - merge=True by default
    - Uses explicit convention Y = i X Z when (a,b)=(1,1)
    """
    if prob_func is None:
        prob_func = make_general_prob('Pr')
    ms = state if isinstance(state, dict) else pure_to_mixed(state, phase_invariant=phase_invariant)
    qbs = tuple(noisy_qubits)
    out = {}
    for pattern in product([0,1], repeat=2*len(qbs)):
        coeff_p = prob_func(*pattern)
        if coeff_p == 0:
            continue
        for key, p_in in ms.items():
            ket = list(key)
            for i,q in enumerate(qbs):
                a = pattern[2*i]
                b = pattern[2*i+1]
                # apply Z^b then X^a so (1,1) gives iXZ = Y by convention
                if b == 1:
                    ket = pauli_z(q, ket)
                if a == 1:
                    ket = pauli_x(q, ket)
            k = _pure_state_key(ket, phase_invariant=phase_invariant)
            out[k] = out.get(k, 0) + p_in * coeff_p
    if merge:
        out = merge_mixed_states(out, phase_invariant=phase_invariant)
    return out


In [ ]:
# Example: 4-qubit protocol with depolarizing noise and F' vs F
# qubits: A=0, B=1, C1=2, C2=3
F = symbols('F', real=True)

phi_plus = create_bell_state(2, 0, 0)
A_B = phi_plus
C1 = create_zero_state(1)
C2 = hadamard(0, create_zero_state(1))
state = comp_st(A_B, C1, C2)
ms = pure_to_mixed(state)

depol = make_iid_depolarizing_prob(F)
ms = pauli_noise(ms, noisy_qubits=[1], prob_func=depol, merge=True)   # noise on B
ms = apply_gate_to_mixed_state(cnot, (0, 2), ms)                      # CNOT(A->C1)
ms = apply_gate_to_mixed_state(cnot, (3, 1), ms)                      # CNOT(C2->B)
ms = pauli_noise(ms, noisy_qubits=[2,3], prob_func=depol, merge=True) # independent noise C1,C2
ms = apply_gate_to_mixed_state(cnot, (3, 0), ms)                      # CNOT(C2->A)
ms = apply_gate_to_mixed_state(cnot, (1, 2), ms)                      # CNOT(B->C1)
ms = apply_gate_to_mixed_state(hadamard, (3,), ms)                    # H(C2)

meas00 = pure_to_mixed(comp_st(create_zero_state(1), create_zero_state(1)))
post = mixed_measurement(meas00, [2,3], ms, merge=True)               # unnormalized by design
success_prob = sum(post.values())

phi_plus_AB = create_bell_state(2, 0, 0)
phi_fraction = fidelity(phi_plus_AB, post)
F_prime = simplify(phi_fraction / success_prob)
print('F_prime(F) =', F_prime)

# Numeric plot on F in [0.5, 1]
import numpy as np
import matplotlib.pyplot as plt
fprime_fn = __import__('sympy').lambdify(F, F_prime, 'numpy')
xs = np.linspace(0.5, 1.0, 200)
ys = fprime_fn(xs)
plt.plot(xs, ys)
plt.xlabel('F')
plt.ylabel("F'")
plt.title("Post-selected fidelity F' vs channel fidelity F")
plt.grid(True)
plt.show()
